# Segment Anything con puntos y cajas

Notebook opcional del **módulo 12** del curso (en la app: `uv run dev` y la ruta `#/modulo/segmentacion-sam`). Usa modelos reales; el módulo interactivo explica la teoría con NumPy.

> **Requisitos:** desde la raíz del repositorio, `uv sync --group notebooks` y abre este notebook con `uv run --group notebooks jupyter lab`. La primera ejecución descarga los modelos de Hugging Face.

In [ ]:
import matplotlib.pyplot as plt
import requests
import torch
from PIL import Image
from transformers import SamModel, SamProcessor

modelo = SamModel.from_pretrained("facebook/sam-vit-base")
proc = SamProcessor.from_pretrained("facebook/sam-vit-base")
url = "https://huggingface.co/ybelkada/segment-anything/resolve/main/assets/car.png"
imagen = Image.open(requests.get(url, stream=True).raw).convert("RGB")

In [ ]:
def segmentar(**prompt):
    x = proc(imagen, return_tensors="pt", **prompt)
    with torch.no_grad():
        y = modelo(**x)
    mascaras = proc.image_processor.post_process_masks(y.pred_masks, x["original_sizes"], x["reshaped_input_sizes"])
    return mascaras[0][0], y.iou_scores[0, 0]

mascaras, iou = segmentar(input_points=[[[450, 600]]])
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for i in range(3):
    ax[i].imshow(imagen)
    ax[i].imshow(mascaras[i], alpha=0.5)
    ax[i].set_title(f"IoU predicho {iou[i]:.2f}")
plt.show()

In [ ]:
mascaras, iou = segmentar(input_boxes=[[[75, 275, 1725, 850]]])
plt.imshow(imagen)
plt.imshow(mascaras[int(iou.argmax())], alpha=0.5)
plt.show()